In [247]:
import glob
import os
import sys
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import tqdm


sys.path.append('..')

from utils.experiment import (
    SimulationConfig,
    OptimizationResult,
    experiment_filename,
    save_experiment,
    load_experiment,
)


result_dirs = [
    '../Time Responsive Policy/Results/*.zip',
    '../Fixed Policy/Results/*.zip',
]

experiment_files = []

for result_dir in result_dirs:
    experiment_files.extend(glob.glob(result_dir))

os.makedirs('./Results', exist_ok=True)


In [248]:
experiment_df = pl.DataFrame(
    [],
    schema={
        'experiment_file': pl.Utf8,
        'initial_wealth': pl.Float64,
        'n_time_nodes': pl.Int64,
        'n_wealth_nodes': pl.Int64,
        'sampler': pl.Utf8,
    }
)


for experiment_file in experiment_files:
    experiment = load_experiment(experiment_file)
    result: OptimizationResult = experiment['result']
    config: SimulationConfig = experiment['config']

    policy = result.best_policy

    experiment_df = experiment_df.vstack(
        pl.DataFrame(
            {
                'experiment_file': [experiment_file],
                'initial_wealth': float(config.INITIAL_WEALTH),
                'n_time_nodes': [config.TIME_NODE_COUNT],
                'n_wealth_nodes': [config.WEALTH_NODE_COUNT],
                'sampler': [config.RETURN_SAMPLER],
            }
        )
    )

experiment_df.write_parquet('./Results/optimisation_experiments.parquet')

In [249]:
# Load best policy data
best_policy_df = pl.DataFrame(
    [],
    schema={
        'experiment_file': pl.Utf8,
        'time_node': pl.Float64,
        'wealth_node': pl.Float64,
        'cash': pl.Float64,
        'bonds': pl.Float64,
        'stocks': pl.Float64,
    }
)

for row in experiment_df.iter_rows(named=True):
    experiment_file = row['experiment_file']
    experiment = load_experiment(experiment_file)
    result: OptimizationResult = experiment['result']
    config: SimulationConfig = experiment['config']

    if config.TIME_NODE_COUNT > 1 and config.WEALTH_NODE_COUNT == 1:
        time_nodes = result.time_nodes
        # old versions of the code may not have time_nodes saved in the result, so we can construct them
        if not time_nodes:
            time_nodes = np.linspace(0, config.SIMULATION_YEARS-1, config.TIME_NODE_COUNT)

        best_policy = result.best_policy
        policy_with_cash = np.zeros((best_policy.shape[0], best_policy.shape[1]+1))
        policy_with_cash[:, 1:] = best_policy
        # cash, bonds, stocks
        policy_with_cash[:, 0] = 1 - np.sum(best_policy, axis=1)
        
        best_policy_df = best_policy_df.vstack(
            pl.DataFrame(
                {
                    'experiment_file': [experiment_file] * len(time_nodes),
                    'time_node': time_nodes,
                    'wealth_node': [None] * len(time_nodes),
                    'cash': policy_with_cash[:, 0],
                    'bonds': policy_with_cash[:, 1],
                    'stocks': policy_with_cash[:, 2],
                }
            )
        )

best_policy_df.write_parquet('./Results/optimisation_best_policies.parquet')

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

outcomes_schema = {
    'experiment_file': pl.Utf8,
    'bankruptcy_probability': pl.Float64,
    'bankruptcy_density': pl.Float64,
    'mean_terminal_wealth': pl.Float64,
    'std_terminal_wealth': pl.Float64,
    'median_terminal_wealth': pl.Float64,
    'p10_terminal_wealth': pl.Float64,
    'p90_terminal_wealth': pl.Float64,
}

def compute_outcome(experiment_file: str) -> dict:
    experiment = load_experiment(experiment_file)
    result: OptimizationResult = experiment['result']

    wealth = result.wealth_simulated
    final_wealth = wealth[:, -1]

    return {
        'experiment_file': experiment_file,
        'bankruptcy_probability': float(np.mean(final_wealth <= 0)),
        'bankruptcy_density': float((wealth <= 0).mean()),
        'mean_terminal_wealth': float(final_wealth.mean()),
        'std_terminal_wealth': float(final_wealth.std()),
        'median_terminal_wealth': float(np.median(final_wealth)),
        'p10_terminal_wealth': float(np.percentile(final_wealth, 10)),
        'p90_terminal_wealth': float(np.percentile(final_wealth, 90)),
    }

experiment_file_list = experiment_df['experiment_file'].to_list()
max_workers = 16
outcome_rows = []

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(compute_outcome, experiment_file) for experiment_file in experiment_file_list]
    for future in tqdm.tqdm(
        as_completed(futures),
        total=len(futures),
        desc=f'Computing outcomes ({max_workers} threads)',
    ):
        outcome_rows.append(future.result())

outcomes_df = pl.DataFrame(outcome_rows, schema=outcomes_schema).sort('experiment_file')
outcomes_df.write_parquet('./Results/optimisation_outcomes.parquet')

Computing outcomes (16 threads):   0%|          | 0/114 [00:00<?, ?it/s]

Computing outcomes (16 threads):   2%|▏         | 2/114 [00:02<01:46,  1.05it/s]